## ML_1034: Rotary Position Embedding (RoPE)

**Difficulty**: Hard | **TorchCode**: #24

### Problem
Implement RoPE which encodes relative position by rotating query/key pairs. Split each embedding into pairs, compute rotation angles based on position and dimension, then apply 2D rotation. Preserves norms and encodes relative distance in dot products.

### Formula
$$\theta_i = \frac{\text{pos}}{10000^{2i/d}}, \quad \begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos\theta_i & -\sin\theta_i \\ \sin\theta_i & \cos\theta_i \end{pmatrix}\begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

In [ ]:
import torch
import math

def apply_rope(q, k):
    B, S, D = q.shape
    pos = torch.arange(S, device=q.device).unsqueeze(1).float()
    dim = torch.arange(0, D, 2, device=q.device).float()
    freqs = 1.0 / (10000.0 ** (dim / D))
    angles = pos * freqs
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)

    def rotate(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos_a - x2 * sin_a,
                            x1 * sin_a + x2 * cos_a], dim=-1).flatten(-2)

    return rotate(q), rotate(k)

In [ ]:
import torch

# --- Test 1: Output shapes ---
q = torch.randn(2, 8, 64); k = torch.randn(2, 8, 64)
q_rot, k_rot = apply_rope(q, k)
assert q_rot.shape == q.shape and k_rot.shape == k.shape

# --- Test 2: Preserves norm ---
torch.manual_seed(0)
q = torch.randn(1, 16, 32); k = torch.randn(1, 16, 32)
q_rot, k_rot = apply_rope(q, k)
assert torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-4)

# --- Test 3: Relative position property ---
torch.manual_seed(0)
q = torch.randn(1, 8, 16); k = torch.randn(1, 8, 16)
q_rot, k_rot = apply_rope(q, k)
q2 = torch.cat([torch.zeros(1, 3, 16), q], dim=1)
k2 = torch.cat([torch.zeros(1, 3, 16), k], dim=1)
q2_rot, k2_rot = apply_rope(q2, k2)
dot1 = (q_rot[:, 0] * k_rot[:, 0]).sum(dim=-1)
dot2 = (q2_rot[:, 3] * k2_rot[:, 3]).sum(dim=-1)
assert torch.allclose(dot1, dot2, atol=1e-4)

# --- Test 4: Gradient flow ---
q = torch.randn(1, 4, 8, requires_grad=True)
k = torch.randn(1, 4, 8, requires_grad=True)
qr, kr = apply_rope(q, k)
(qr.sum() + kr.sum()).backward()
assert q.grad is not None and k.grad is not None

print('All tests passed!')